<a href="https://colab.research.google.com/github/mohamedalaaaz/testpytroch/blob/main/AL%20payload%20attack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
import re

# Load dataset
df = pd.read_csv("payloads.csv")

# Clean payloads
def clean_text(text):
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # Remove non-ASCII
    return text.lower()

df['payload'] = df['payload'].apply(clean_text)

# Tokenization
def tokenize(text):
    return text.split()

df['tokens'] = df['payload'].apply(tokenize)

# Build vocabulary
vocab = Counter()
for tokens in df['tokens']:
    vocab.update(tokens)

word2idx = {word: idx + 2 for idx, (word, _) in enumerate(vocab.items())}
word2idx['<PAD>'] = 0
word2idx['<UNK>'] = 1

def encode(tokens):
    return [word2idx.get(token, word2idx['<UNK>']) for token in tokens]

df['encoded'] = df['tokens'].apply(encode)


In [ ]:
class PayloadDataset(Dataset):
    def __init__(self, data):
        self.X = data['encoded']
        self.y = data['label'].values

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx]), torch.tensor(self.y[idx])

def collate_fn(batch):
    X, y = zip(*batch)
    X_pad = pad_sequence(X, batch_first=True, padding_value=0)
    return X_pad, torch.tensor(y)

train_data, test_data = train_test_split(df, test_size=0.2, stratify=df['label'])

train_ds = PayloadDataset(train_data)
test_ds = PayloadDataset(test_data)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=32, collate_fn=collate_fn)


In [ ]:
import torch.nn as nn

class WebAttackDetector(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        out = self.fc(hidden[-1])
        return self.sigmoid(out).squeeze()


In [ ]:
model = WebAttackDetector(vocab_size=len(word2idx))
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Training loop
for epoch in range(5):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.float().to(device)
        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        preds = model(X_batch).cpu().numpy()
        y_pred.extend(preds > 0.5)
        y_true.extend(y_batch.numpy())

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1 Score:", f1_score(y_true, y_pred))


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        preds = model(X_batch).cpu().numpy()
        y_pred.extend(preds > 0.5)
        y_true.extend(y_batch.numpy())

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1 Score:", f1_score(y_true, y_pred))
